# 10. EuroMart quarterly snapshots

LightGCN is evaluated on EuroMart one quarter ahead, but the shared Run 2 release only contains monthly snapshots. This notebook builds the quarterly snapshots with the same builder that produced the monthly release; the only change is that the period length is a parameter.

1. Define the builder
2. Check that it reproduces the monthly release exactly (EuroMart 48 snapshots, Retail POS 12)
3. Build the quarterly snapshots into `data/euromart_quarterly/`
4. Compare them with the reference build

Inputs are read from `data/inputs/`.

In [1]:
import json, hashlib, pathlib
import numpy as np
import pandas as pd

SUB  = pathlib.Path.cwd()                 # .../SC4020-DataMining/Submission
REPO = SUB.parent
INP  = SUB / "data" / "inputs"
REF_MONTHLY = {"euromart": SUB / "data" / "euromart_monthly",              # team v3 release
               "retail_pos": REPO / "graph_data" / "retail_pos" / "monthly"}  # team v3 release
OUT_Q = SUB / "data" / "euromart_quarterly"
OUT_Q.mkdir(parents=True, exist_ok=True)
print({k: len(list(v.glob('*.npz'))) for k, v in REF_MONTHLY.items()})

{'euromart': 48, 'retail_pos': 12}


## 1. Snapshot builder
The builder follows the shared Run 2 pipeline, with the period frequency ("M" monthly or "Q" quarterly) as an argument. Each snapshot holds the graph built from purchases strictly before the target period, the eligible customers and their labels for that period.

In [2]:
"""
Period-generalised re-implementation of the team's Run 2 snapshot builder
(02_Shared_Graph_Pipeline_v3.ipynb, cells 11-32).

Logic is transcribed verbatim from that notebook; the ONLY generalisation is
that the period frequency ("M" or "Q") is a parameter instead of hard-coded "M".
Validated by rebuilding the 60 shipped monthly snapshots bit-for-bit.
"""
import json, hashlib
import numpy as np
import pandas as pd

CUSTOMER_FEATURE_NAMES = ["log_interactions","log_orders","log_quantity","log_value",
                          "category_diversity_fraction","active_month_fraction","freshness"]
CATEGORY_FEATURE_NAMES = ["log_interactions","log_orders","log_quantity","log_value",
                          "log_unique_customers","popularity_share","freshness"]
EDGE_FEATURE_NAMES     = ["log_interactions","log_orders","log_quantity","log_value","freshness"]


def period_from_string(x, freq):
    return pd.Period(str(x), freq=freq)


def freshness_from_last_period(last_p, target_p, freq):
    last_p = period_from_string(last_p, freq)
    target_p = period_from_string(target_p, freq)
    complete_inactive = max(target_p.ordinal - last_p.ordinal - 1, 0)
    return 1.0 / (1.0 + complete_inactive)


def fit_zscore(rows, n_features):
    if not rows:
        return {"mean": np.zeros(n_features, dtype=np.float64),
                "std":  np.ones(n_features, dtype=np.float64)}
    x = np.concatenate(rows, axis=0).astype(np.float64)
    mean = x.mean(axis=0)
    std = x.std(axis=0)
    std = np.where(std < 1e-12, 1.0, std)
    return {"mean": mean, "std": std}


def apply_zscore(matrix, scaler, active_mask=None):
    x = (matrix.astype(np.float64) - scaler["mean"]) / scaler["std"]
    if active_mask is not None:
        x = x.copy()
        x[~active_mask] = 0.0
    return x.astype(np.float32)


def serialise_scaler(scaler, names):
    return {"feature_names": list(names),
            "mean": scaler["mean"].tolist(),
            "std": scaler["std"].tolist()}


def build_registries(frame, category_registry_csv, freq, last_train_target):
    """Mirror of CELL 13, generalised over freq."""
    temp = frame.copy()
    temp["period"] = pd.PeriodIndex(temp["period_str"], freq=freq)

    cr = temp.groupby("customer_id", as_index=False).agg(
        first_seen_period=("period", "min"), last_seen_period=("period", "max"))
    cr["customer_id"] = cr["customer_id"].astype(str)
    cr["first_seen_period"] = cr["first_seen_period"].astype(str)
    cr["last_seen_period"] = cr["last_seen_period"].astype(str)
    cr = cr.sort_values(["first_seen_period", "customer_id"], kind="mergesort").reset_index(drop=True)
    cr["customer_idx"] = np.arange(len(cr), dtype=np.int64)
    cr["node_idx"] = cr["customer_idx"]

    run0 = pd.read_csv(category_registry_csv)
    assert (run0["category_idx"].to_numpy() == np.arange(17)).all()
    assert set(run0["category"].astype(str)) == set(frame["category"].astype(str))
    kr = run0[["category_idx", "category"]].copy()
    kr["category_idx"] = kr["category_idx"].astype(np.int64)
    kr["node_idx"] = len(cr) + kr["category_idx"]
    cat_first = temp.groupby("category")["period"].min().astype(str)
    kr["first_seen_period"] = kr["category"].map(cat_first)

    last_train_p = period_from_string(last_train_target, freq)
    cr["seen_in_any_train_input_graph"] = pd.PeriodIndex(cr["first_seen_period"], freq=freq) < last_train_p
    kr["seen_in_any_train_input_graph"] = pd.PeriodIndex(kr["first_seen_period"], freq=freq) < last_train_p
    return {"customers": cr, "categories": kr}


def build_raw_snapshot(frame, dataset_key, target_period_str, customer_registry,
                       category_registry, freq, split_lookup):
    """Verbatim transcription of CELL 16's build_raw_snapshot, freq-generalised."""
    target_period = period_from_string(target_period_str, freq)
    temp = frame.copy()
    temp["period"] = pd.PeriodIndex(temp["period_str"], freq=freq)
    history = temp.loc[temp["period"] < target_period].copy()
    target = temp.loc[temp["period"] == target_period].copy()
    assert (history["period"] < target_period).all()
    assert (target["period"] == target_period).all()

    customer_to_idx = dict(zip(customer_registry["customer_id"].astype(str),
                               customer_registry["customer_idx"].astype(int)))
    category_to_idx = dict(zip(category_registry["category"].astype(str),
                               category_registry["category_idx"].astype(int)))
    n_customers = len(customer_registry)
    n_categories = len(category_registry)
    assert n_categories == 17

    # ---- customer features ----
    customer_features_raw = np.zeros((n_customers, len(CUSTOMER_FEATURE_NAMES)), dtype=np.float32)
    history_customer_mask = np.zeros(n_customers, dtype=bool)
    if len(history):
        cagg = history.groupby("customer_id").agg(
            interactions=("category", "size"), orders=("order_id", "nunique"),
            quantity=("quantity", "sum"), value=("value_proxy", "sum"),
            distinct_categories=("category", "nunique"), active_periods=("period", "nunique"),
            first_period=("period", "min"), last_period=("period", "max"))
        for customer_id, row in cagg.iterrows():
            idx = customer_to_idx[str(customer_id)]
            historical_window = max(target_period.ordinal - row["first_period"].ordinal, 1)
            active_fraction = row["active_periods"] / historical_window
            customer_features_raw[idx] = np.asarray([
                np.log1p(row["interactions"]), np.log1p(row["orders"]),
                np.log1p(row["quantity"]), np.log1p(row["value"]),
                row["distinct_categories"] / n_categories, active_fraction,
                freshness_from_last_period(row["last_period"], target_period, freq)],
                dtype=np.float32)
            history_customer_mask[idx] = True

    # ---- category features ----
    category_features_raw = np.zeros((n_categories, len(CATEGORY_FEATURE_NAMES)), dtype=np.float32)
    history_category_mask = np.zeros(n_categories, dtype=bool)
    if len(history):
        total_history_rows = len(history)
        kagg = history.groupby("category").agg(
            interactions=("customer_id", "size"), orders=("order_id", "nunique"),
            quantity=("quantity", "sum"), value=("value_proxy", "sum"),
            unique_customers=("customer_id", "nunique"), last_period=("period", "max"))
        for category, row in kagg.iterrows():
            idx = category_to_idx[str(category)]
            category_features_raw[idx] = np.asarray([
                np.log1p(row["interactions"]), np.log1p(row["orders"]),
                np.log1p(row["quantity"]), np.log1p(row["value"]),
                np.log1p(row["unique_customers"]),
                row["interactions"] / total_history_rows,
                freshness_from_last_period(row["last_period"], target_period, freq)],
                dtype=np.float32)
            history_category_mask[idx] = True

    # ---- cumulative binary graph ----
    edge_index_bipartite = np.empty((2, 0), dtype=np.int64)
    edge_index = np.empty((2, 0), dtype=np.int64)
    edge_attr_raw = np.empty((0, len(EDGE_FEATURE_NAMES)), dtype=np.float32)
    edge_attr_raw_bidir = np.empty((0, len(EDGE_FEATURE_NAMES)), dtype=np.float32)
    pair_agg = None
    if len(history):
        pair_agg = history.groupby(["customer_id", "category"], as_index=False).agg(
            interactions=("order_id", "size"), orders=("order_id", "nunique"),
            quantity=("quantity", "sum"), value=("value_proxy", "sum"),
            last_period=("period", "max"))
        cust_idx = pair_agg["customer_id"].astype(str).map(customer_to_idx).to_numpy(dtype=np.int64)
        cat_idx = pair_agg["category"].astype(str).map(category_to_idx).to_numpy(dtype=np.int64)
        edge_index_bipartite = np.vstack([cust_idx, cat_idx])
        cat_node_idx = n_customers + cat_idx
        forward = np.vstack([cust_idx, cat_node_idx])
        reverse = np.vstack([cat_node_idx, cust_idx])
        edge_index = np.concatenate([forward, reverse], axis=1)
        pair_freshness = np.asarray(
            [freshness_from_last_period(x, target_period, freq) for x in pair_agg["last_period"]],
            dtype=np.float32)
        edge_attr_raw = np.column_stack([
            np.log1p(pair_agg["interactions"].to_numpy(dtype=float)),
            np.log1p(pair_agg["orders"].to_numpy(dtype=float)),
            np.log1p(pair_agg["quantity"].to_numpy(dtype=float)),
            np.log1p(pair_agg["value"].to_numpy(dtype=float)),
            pair_freshness]).astype(np.float32)
        edge_attr_raw_bidir = np.concatenate([edge_attr_raw, edge_attr_raw], axis=0)

    # ---- degrees ----
    customer_history_degree = np.zeros(n_customers, dtype=np.int16)
    category_history_degree = np.zeros(n_categories, dtype=np.int32)
    if edge_index_bipartite.shape[1]:
        customer_history_degree = np.bincount(edge_index_bipartite[0], minlength=n_customers).astype(np.int16)
        category_history_degree = np.bincount(edge_index_bipartite[1], minlength=n_categories).astype(np.int32)

    # ---- eligibility + labels ----
    historical_customers = set(history["customer_id"].astype(str))
    active_target_customers = set(target["customer_id"].astype(str))
    eligible_ids = sorted(historical_customers & active_target_customers,
                          key=lambda x: customer_to_idx[x])
    eligible_customer_idx = np.asarray([customer_to_idx[x] for x in eligible_ids], dtype=np.int64)
    labels = np.zeros((len(eligible_ids), n_categories), dtype=np.uint8)
    eligible_row = {cid: i for i, cid in enumerate(eligible_ids)}
    if len(eligible_ids):
        target_pairs = target[["customer_id", "category"]].drop_duplicates()
        for row in target_pairs.itertuples(index=False):
            cid = str(row.customer_id)
            if cid not in eligible_row:
                continue
            labels[eligible_row[cid], category_to_idx[str(row.category)]] = 1
        assert (labels.sum(axis=1) >= 1).all()

    # ---- dense pair-history matrices ----
    shape = (len(eligible_ids), n_categories)
    eligible_history_pair_mask = np.zeros(shape, dtype=np.uint8)
    eligible_previous_month_mask = np.zeros(shape, dtype=np.uint8)
    eligible_history_interaction_count = np.zeros(shape, dtype=np.int32)
    eligible_history_order_count = np.zeros(shape, dtype=np.int32)
    eligible_history_quantity = np.zeros(shape, dtype=np.float32)
    eligible_history_value = np.zeros(shape, dtype=np.float32)
    eligible_history_freshness = np.zeros(shape, dtype=np.float32)
    if pair_agg is not None and len(eligible_ids):
        eligible_set = set(eligible_ids)
        for row in pair_agg.itertuples(index=False):
            cid = str(row.customer_id)
            if cid not in eligible_set:
                continue
            r = eligible_row[cid]
            c = category_to_idx[str(row.category)]
            fresh = freshness_from_last_period(row.last_period, target_period, freq)
            eligible_history_pair_mask[r, c] = 1
            eligible_previous_month_mask[r, c] = int(
                period_from_string(row.last_period, freq) == (target_period - 1))
            eligible_history_interaction_count[r, c] = int(row.interactions)
            eligible_history_order_count[r, c] = int(row.orders)
            eligible_history_quantity[r, c] = float(row.quantity)
            eligible_history_value[r, c] = float(row.value)
            eligible_history_freshness[r, c] = float(fresh)

    eligible_history_degree = (customer_history_degree[eligible_customer_idx]
                               if len(eligible_customer_idx) else np.zeros(0, dtype=np.int16))
    train_seen_customer_mask = customer_registry["seen_in_any_train_input_graph"].to_numpy(dtype=bool)
    train_seen_category_mask = category_registry["seen_in_any_train_input_graph"].to_numpy(dtype=bool)
    eligible_train_seen_customer_mask = (train_seen_customer_mask[eligible_customer_idx]
                                         if len(eligible_customer_idx) else np.zeros(0, dtype=bool))

    n_hist_c = int(history_customer_mask.sum())
    n_hist_k = int(history_category_mask.sum())
    n_edges = int(edge_index_bipartite.shape[1])
    fixed_den = n_hist_c * n_categories
    seen_den = n_hist_c * n_hist_k
    return {
        "dataset": dataset_key, "target_month": str(target_period),
        "split": split_lookup(target_period_str),
        "history_rows": int(len(history)), "target_rows": int(len(target)),
        "customer_features_raw": customer_features_raw,
        "category_features_raw": category_features_raw,
        "history_customer_mask": history_customer_mask,
        "history_category_mask": history_category_mask,
        "edge_index_bipartite": edge_index_bipartite, "edge_index": edge_index,
        "edge_attr_raw": edge_attr_raw, "edge_attr_raw_bidir": edge_attr_raw_bidir,
        "customer_history_degree": customer_history_degree,
        "category_history_degree": category_history_degree,
        "eligible_customer_idx": eligible_customer_idx, "labels": labels,
        "eligible_history_pair_mask": eligible_history_pair_mask,
        "eligible_previous_month_mask": eligible_previous_month_mask,
        "eligible_history_interaction_count": eligible_history_interaction_count,
        "eligible_history_order_count": eligible_history_order_count,
        "eligible_history_quantity": eligible_history_quantity,
        "eligible_history_value": eligible_history_value,
        "eligible_history_freshness": eligible_history_freshness,
        "eligible_history_degree": eligible_history_degree,
        "train_seen_customer_mask": train_seen_customer_mask,
        "train_seen_category_mask": train_seen_category_mask,
        "eligible_train_seen_customer_mask": eligible_train_seen_customer_mask,
        "historical_customers": n_hist_c, "historical_categories": n_hist_k,
        "unique_edges": n_edges,
        "density_fixed_17": n_edges / fixed_den if fixed_den else 0.0,
        "density_seen_categories": n_edges / seen_den if seen_den else 0.0,
        "mean_customer_degree": n_edges / n_hist_c if n_hist_c else 0.0,
    }


def fit_and_apply_scalers(snapshots):
    """CELL 24 + CELL 26."""
    crows, krows, erows = [], [], []
    for s in snapshots:
        if s["split"] != "train":
            continue
        if s["history_customer_mask"].any():
            crows.append(s["customer_features_raw"][s["history_customer_mask"]])
        if s["history_category_mask"].any():
            krows.append(s["category_features_raw"][s["history_category_mask"]])
        if len(s["edge_attr_raw"]):
            erows.append(s["edge_attr_raw"])
    scalers = {"customer": fit_zscore(crows, len(CUSTOMER_FEATURE_NAMES)),
               "category": fit_zscore(krows, len(CATEGORY_FEATURE_NAMES)),
               "edge":     fit_zscore(erows, len(EDGE_FEATURE_NAMES))}
    for s in snapshots:
        s["customer_features"] = apply_zscore(s["customer_features_raw"], scalers["customer"],
                                              s["history_customer_mask"])
        s["category_features"] = apply_zscore(s["category_features_raw"], scalers["category"],
                                              s["history_category_mask"])
        s["edge_attr"] = apply_zscore(s["edge_attr_raw"], scalers["edge"])
        s["edge_attr_bidir"] = apply_zscore(s["edge_attr_raw_bidir"], scalers["edge"])
    return scalers


def loss_diagnostics(snapshots):
    """CELL 28 - the shared pos_weight."""
    train_labels = [s["labels"] for s in snapshots if s["split"] == "train" and s["labels"].size]
    y = np.concatenate(train_labels, axis=0)
    pos = int(y.sum()); cand = int(y.size); neg = cand - pos
    return {"train_eligible_customer_months": int(y.shape[0]),
            "train_positive_pairs": pos, "train_negative_pairs": neg,
            "train_candidate_pairs": cand,
            "train_positive_prevalence": float(pos / cand),
            "primary_scalar_pos_weight": float(neg / pos if pos else 1.0)}


SAVE_KEYS = ["edge_index_bipartite","edge_index","edge_attr_raw","edge_attr","edge_attr_bidir",
             "customer_features_raw","customer_features","category_features_raw","category_features",
             "history_customer_mask","history_category_mask","customer_history_degree",
             "category_history_degree","eligible_customer_idx","labels",
             "eligible_history_pair_mask","eligible_previous_month_mask",
             "eligible_history_interaction_count","eligible_history_order_count",
             "eligible_history_quantity","eligible_history_value","eligible_history_freshness",
             "eligible_history_degree","train_seen_customer_mask","train_seen_category_mask",
             "eligible_train_seen_customer_mask"]


def save_snapshot(snapshot, path):
    np.savez_compressed(path, **{k: snapshot[k] for k in SAVE_KEYS})


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


## 2. Check: rebuild the monthly release
Before using the builder for quarters, rebuild the existing monthly snapshots and compare every array with the shared release. All arrays must match.

In [3]:
CONTRACT = json.load(open(INP / "project_contract_v3.json"))

def load_frame(ds):
    f = pd.read_csv(INP / f"{ds}_transactions_v3.csv")
    f["category"] = f["category"].astype(str)     # customer_id keeps its native dtype, as the team grouped on it
    f["period_str"] = f["month"].astype(str)
    return f

report = {}
for ds in ["euromart", "retail_pos"]:
    sm = CONTRACT["datasets"][ds]["split_months"]
    split_of = {m: k for k in ("train", "val", "test") for m in sm[k]}
    periods = sm["train"] + sm["val"] + sm["test"]
    frame = load_frame(ds)
    assert sorted(frame["period_str"].unique()) == sorted(periods)
    reg = build_registries(frame, INP / f"{ds}_category_registry.csv", "M", sm["train"][-1])
    snaps = [build_raw_snapshot(frame, ds, p, reg["customers"], reg["categories"],
                                "M", lambda x: split_of[x]) for p in periods]
    fit_and_apply_scalers(snaps)
    nmis, ntot, worst = 0, 0, 0.0
    for s in snaps:
        ref = np.load(REF_MONTHLY[ds] / f"{s['target_month']}.npz", allow_pickle=True)
        for k in SAVE_KEYS:
            ntot += 1
            a, b = s[k], ref[k]
            if a.shape != b.shape or a.dtype != b.dtype:
                nmis += 1; continue
            if np.issubdtype(a.dtype, np.floating):
                d = float(np.max(np.abs(a.astype(np.float64) - b.astype(np.float64)))) if a.size else 0.0
                worst = max(worst, d); nmis += d > 0
            else:
                nmis += not np.array_equal(a, b)
    report[ds] = dict(snapshots=len(snaps), arrays_checked=ntot, mismatched_arrays=int(nmis),
                      max_float_abs_diff=worst)
    print(f"{ds}: {len(snaps)} snapshots | {ntot} arrays | mismatches {nmis} | max|diff| {worst:.1e}")

GATE = all(r["mismatched_arrays"] == 0 for r in report.values())
print("\nGATE:", "PASS" if GATE else "FAIL")
assert GATE

euromart: 48 snapshots | 1248 arrays | mismatches 0 | max|diff| 0.0e+00


retail_pos: 12 snapshots | 312 arrays | mismatches 0 | max|diff| 0.0e+00

GATE: PASS


## 3. Quarterly build
Train 2011Q2-2013Q3, validation 2013Q4-2014Q1, test 2014Q2-2014Q4. 2011Q1 is history only (its snapshot is empty), the same role as 2011-01 in the monthly setup.

In [4]:
TRAIN = ["2011Q1","2011Q2","2011Q3","2011Q4","2012Q1","2012Q2","2012Q3","2012Q4",
         "2013Q1","2013Q2","2013Q3"]
VAL   = ["2013Q4","2014Q1"]
TEST  = ["2014Q2","2014Q3","2014Q4"]
SPLIT_OF = {**{p: "train" for p in TRAIN}, **{p: "val" for p in VAL}, **{p: "test" for p in TEST}}
PERIODS = TRAIN + VAL + TEST

frame = pd.read_csv(INP / "euromart_transactions_v3.csv")
frame["category"] = frame["category"].astype(str)
frame["period_str"] = pd.PeriodIndex(frame["month"].astype(str), freq="M").asfreq("Q").astype(str)
assert sorted(frame["period_str"].unique()) == sorted(PERIODS)

reg = build_registries(frame, INP / "euromart_category_registry.csv", "Q", "2013Q3")
snaps = [build_raw_snapshot(frame, "euromart", p, reg["customers"], reg["categories"],
                            "Q", lambda x: SPLIT_OF[x]) for p in PERIODS]
scalers = fit_and_apply_scalers(snaps)
loss = loss_diagnostics(snaps)

rows = []
for s in snaps:
    p = OUT_Q / f"{s['target_month']}.npz"
    save_snapshot(s, p)
    lab = s["labels"]
    rows.append({"dataset": "euromart_quarterly", "period": s["target_month"], "split": s["split"],
                 "history_rows": s["history_rows"], "target_rows": s["target_rows"],
                 "eligible_customers": len(s["eligible_customer_idx"]),
                 "positive_pairs": int(lab.sum()), "candidate_pairs": int(lab.size),
                 "repeated_positive_pairs": int((lab.astype(bool) & s["eligible_history_pair_mask"].astype(bool)).sum()),
                 "prev_period_positive_pairs": int((lab.astype(bool) & s["eligible_previous_month_mask"].astype(bool)).sum()),
                 "unique_edges": s["unique_edges"], "historical_customers": s["historical_customers"],
                 "density_fixed_17": s["density_fixed_17"], "sha256": sha256_file(p)})
idx = pd.DataFrame(rows)
idx.to_csv(OUT_Q / "snapshot_index.csv", index=False)
reg["customers"].to_csv(OUT_Q / "customer_registry.csv", index=False)
reg["categories"].to_csv(OUT_Q / "category_registry.csv", index=False)
json.dump({"contract_version": "3.0", "scaling": "zscore",
           "fitted_on": "historically observed node/edge rows from training input snapshots only",
           "customer": serialise_scaler(scalers["customer"], CUSTOMER_FEATURE_NAMES),
           "category": serialise_scaler(scalers["category"], CATEGORY_FEATURE_NAMES),
           "edge": serialise_scaler(scalers["edge"], EDGE_FEATURE_NAMES)},
          open(OUT_Q / "feature_scalers.json", "w"), indent=2)
json.dump(loss, open(OUT_Q / "loss_diagnostics.json", "w"), indent=2)

print("pos_weight:", round(loss["primary_scalar_pos_weight"], 6),
      "| prevalence:", round(loss["train_positive_prevalence"], 6))
print("customers:", len(reg["customers"]), "| categories:", len(reg["categories"]))
idx[["period", "split", "eligible_customers", "positive_pairs", "unique_edges"]]

pos_weight: 7.099659 | prevalence: 0.123462
customers: 792 | categories: 17


,period,split,eligible_customers,positive_pairs,unique_edges
0,2011Q1,train,0,0,0
1,2011Q2,train,21,49,218
2,2011Q3,train,67,135,536
3,2011Q4,train,88,176,901
4,2012Q1,train,94,194,1273
5,2012Q2,train,119,251,1522
6,2012Q3,train,187,390,1826
7,2012Q4,train,213,450,2273
8,2013Q1,train,165,302,2656
9,2013Q2,train,187,397,2919


## 4. Check against the reference build
The array contents are hashed rather than the `.npz` files, so the comparison does not depend on how the archive was written.

In [5]:
def content_hash(path):
    z = np.load(path, allow_pickle=True); h = hashlib.sha256()
    for k in sorted(z.files):
        a = np.ascontiguousarray(z[k])
        h.update(k.encode()); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

ref = pd.read_csv(SUB / "data" / "reference" / "euromart_quarterly_content_hashes_original.csv")
built = pd.DataFrame([dict(period=p, content_sha256=content_hash(OUT_Q / f"{p}.npz")) for p in PERIODS])
chk = built.merge(ref, on="period", suffixes=("_built", "_reported"))
chk["identical"] = chk.content_sha256_built == chk.content_sha256_reported
print(f"{int(chk.identical.sum())} of {len(chk)} quarterly snapshots are identical, array by array, to the reference build")
assert chk.identical.all() and len(chk) == len(PERIODS)
chk[["period", "identical"]]

16 of 16 quarterly snapshots are identical, array by array, to the reference build


,period,identical
0,2011Q1,True
1,2011Q2,True
2,2011Q3,True
3,2011Q4,True
4,2012Q1,True
5,2012Q2,True
6,2012Q3,True
7,2012Q4,True
8,2013Q1,True
9,2013Q2,True
